# Whisper Fine-Tuning — Duration Sweep
--------

This notebook runs a series of fine-tuning experiments on progressively larger nested
training sets (1 h → 10 h) to measure how dataset size affects ASR performance.
It relies on helper modules in `src/train/` and `src/data/` to keep the experiment
loop clean and readable.

### Helper functions

**`src/train/whisper_duration_experiment.py`**
- `load_model_and_processor` — loads a fresh Whisper model and processor from the Hub before each run so every experiment starts from the same base weights
- `prepare_train_dataset` — reads a duration manifest, splits into train/validation by audio duration, and maps audio + transcripts to Whisper input features
- `prepare_test_dataset` — same feature extraction for the held-out test set; called once and reused across all runs since log-mel features are model-agnostic
- `run_training` — configures the `Seq2SeqTrainer` with the data collator, training arguments, and WER/CER metric computation, then runs fine-tuning
- `run_evaluation` — runs inference on the held-out test set, computes WER and CER, and saves a per-run predictions CSV

**`src/train/train_whisper.py`**
- `load_config` — loads the YAML hyperparameter config
- `build_training_args` — constructs `Seq2SeqTrainingArguments` from the config
- `DataCollatorSpeechSeq2SeqWithPadding` — pads input features and labels for batch training
- `compute_asr_metrics` — decodes predictions and computes WER and CER during evaluation steps

**`src/data/data_utils.py`**
- `load_audio_data` — reads a CSV manifest, resolves audio paths, and returns a Hugging Face
`DatasetDict` with train/validation splits or a single `Dataset` for test sets

## 1. Python Setup

In [ ]:
ENV = "local"  # options: "local", "colab"

In [ ]:
import os
import time
from datetime import datetime
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import login

# ── Project path ──────────────────────────────────────────────────────────────
if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_SRC = Path("/content/drive/MyDrive/chichewa-asr/src")
else:
    # Modify index at the end as required to point to the parent directory of `src/` (which contains the `src` package)
    PROJECT_SRC = Path().cwd().parents[1]

sys.path.append(str(PROJECT_SRC))

from src.train.train_whisper import load_config
from src.train.whisper_duration_experiment import (
    load_model_and_processor,
    prepare_train_dataset,
    prepare_test_dataset,
    run_training,
    run_evaluation,
)

## 2. Paths and Global Configuration

In [ ]:
# ==========================================
# 1. BASE WORKING DIRECTORY AND PATHS
# ==========================================
# Base directory of the project (one level up from src/)
DIR_BASE   = Path.cwd().parents[1]
DIR_DATA   = DIR_BASE / "data"

# Hold out test set 
DIR_TEST               = DIR_DATA / "test"
FILE_MANIFEST_TEST     = DIR_TEST / "metadata.csv"

# Audio files for the dev set and the dev set with nested duration
DIR_DEV                     = DIR_DATA / "dev"
DIR_DEV_NESTED_DURATION     = DIR_DATA / "dev_nested_duration"

# Hyperparameter config file path
FILE_CONFIG = DIR_BASE / "configs" / "whisper_params.yaml"

# Outputs directory where we keep the results of the experiments
DIR_OUTPUTS = DIR_BASE / "outputs"
DIR_RESULTS = DIR_OUTPUTS / "duration_exp_whisper_small"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

# Model checkpoint directory (for saving model checkpoints during training)
DIR_MODEL_CHECKPOINTS = DIR_BASE / "models/checkpoints"
DIR_MODEL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. HARDWARE SETUP
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
# Human-readable label for the machine/device used — update as needed
DEVICE_LABEL = "mps-96gb-ram"  

In [ ]:
# ==========================================
# 3. LOGIN TO HUGGINGFACE HUB
# ==========================================
if ENV == "colab":
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
else:
    load_dotenv()
    login(token=os.getenv("HF_TOKEN"))

## 4. Load Base Config and Held-Out Test Set

These two objects are created once and shared across every duration run.
The test set is preprocessed with a temporary processor (base model weights);
audio features are model-agnostic so this is correct for all runs.

In [ ]:
# Load base config once
base_config = load_config(FILE_CONFIG)
print("Config loaded:", FILE_CONFIG)

# Pre-process the held-out test set once — reused across all runs.
# prepare_test_dataset builds its own processor internally from base_config.
dataset_test = prepare_test_dataset(
    FILE_MANIFEST_TEST,
    audio_dir=DIR_TEST,
    base_config=base_config,
    audio_fname_col="audio_filename",
    duration_col="duration_seconds",
)

print(f"\nHeld-out test set ready: {len(dataset_test):,} utterances")


## 5. Define Duration Datasets 

Provide paths to all target nested duration manifest files. 

In [ ]:
# ================================
# BUILD DURATION_DATASETS 
# ===============================
DURATION_DATASETS = {
    f"{h}h": DIR_DEV_NESTED_DURATION / f"train_{h}h.csv"
    for h in range(1, 3)
}

# Sanity-check that all manifests exist before launching long runs
missing = [str(p) for p in DURATION_DATASETS.values() if not Path(p).exists()]
if missing:
    print("WARNING — manifest files not found:")
    for m in missing:
        print(f"  {m}")
else:
    print(f"All {len(DURATION_DATASETS)} manifest files found. Ready to run.")

## 6. Run Fine-tuning 

Each iteration: loads a fresh base model → trains → (optionally) pushes to Hub
→ evaluates on the shared test set → saves predictions CSV.

In [ ]:
# ==========================================
# EXPERIMENT SETTINGS
# ==========================================
model_id     = base_config["model"]["model_name_or_path"]  
model_name   = model_id.split("/")[-1]                     
HUB_MODEL_ID = f"dmatekenya/{model_name}-chichewa"         

summary = []

for duration_label, manifest_path in DURATION_DATASETS.items():
    if not manifest_path.exists():
        print(f"Skipping {duration_label} — manifest not found.")
        continue

    print(f"\n{'='*60}\n  EXPERIMENT: {duration_label}\n{'='*60}")

    hub_model_id = f"{HUB_MODEL_ID}-{duration_label}"
    output_dir   = DIR_MODEL_CHECKPOINTS / f"{model_name}-chichewa-{duration_label}"

    # 1. load model and processor
    model, processor = load_model_and_processor(base_config)

    # 2. prepare training data
    dataset_train = prepare_train_dataset(manifest_path, DIR_DEV, processor)

    # 3. Train model 
    train_start = time.time()
    trainer = run_training(model, processor, dataset_train, base_config, hub_model_id, output_dir)
    train_minutes = (time.time() - train_start) / 60

    # 4. Push to Hub
    # ==========================================
    # print(f"  Pushing to Hub: {hub_model_id}")
    # trainer.push_to_hub()

    # 5. Evaluate on held-out test set
    row = run_evaluation(model, processor, dataset_test, duration_label, DIR_RESULTS, model_id=hub_model_id, debug=True)
    summary.append({
    "run_timestamp":    datetime.now().strftime("%Y-%m-%d %H:%M"),
    "duration":         duration_label,
    "wer":              row["wer"],
    "cer":              row["cer"],
    "hub_model_id":     hub_model_id,
    "device":           DEVICE_LABEL,
    "train_minutes":    round(train_minutes, 2),
    })
    # Save rolling summary after every run so partial results survive a crash
    pd.DataFrame(summary).to_csv(DIR_RESULTS / "duration_sweep_summary.csv", index=False)

## 7. Inspect Summary Results 

In [ ]:
df_summary = pd.DataFrame(summary)
print("\nFinal summary of duration sweep:")
display(df_summary.style.format({"wer": "{:.2f}%", "cer": "{:.2f}%"}))
